#  MyAnimeList (MAL) Adathalmaz Feltáró Elemzése
## Minőség, rajongás vagy puszta hype? – A sikeres animék anatómiája

### A projekt célkitűzése és kutatási kérdése
A közösségi média és a tartalomfogyasztó platformok korában egy-egy alkotás sikerét számos különböző mutatóval mérhetjük. Jelen elemzés központi kérdése, hogy **mást mutat-e egy anime tényleges minőségéről és megítéléséről az átlagos értékelés (`Score`), a kedvencek száma (`Favorites`) és a nézői érdeklődés (`Members`)?** 

A kutatás során nemcsak e három fő dimenzió közötti korrelációkat és a toplisták átfedéseit vizsgáljuk meg, hanem felkutatjuk a feltűnően eltérő eseteket (outliereket) is: a túlértékelt közönségkedvenceket, a rétegközönségnek szóló kritikai remekműveket, valamint a kultikus klasszikusokat. Emellett górcső alá vesszük, hogy a megfigyelt mintázatok milyen összefüggésben állnak az animék **típusával (`Type`)**, **megjelenési korával (`Aired` / `Premiered`)**, **epizódszámával (`Episodes`)** és **forrásanyagával (`Source`)**.

---

### Az adathalmazok bemutatása
Az elemzés két, egymást kiegészítő, nagyméretű MyAnimeList (MAL) adathalmazra épül:

1. **`anime.csv` (Metaadatok és aggregált statisztikák):**
   * **Méret:** 17 562 egyedi anime részletes adatlapja.
   * **Tartalom:** Általános információk (cím, típus, epizódszám, stúdió, forrásanyag, műfajok, korhatár-besorolás), közösségi aktivitási mutatók (`Members`, `Favorites`, `Watching`, `Completed`, `Dropped`, `Plan to Watch`), valamint a pontszámok részletes eloszlása (`Score`, illetve a `Score-1`-től `Score-10`-ig terjedő szavazatszámok).
2. **`animelist.csv` (Felhasználói interakciók – Big Data):**
   * **Méret:** ~109 millió sor, amely 325 772 egyedi felhasználó értékeléseit és naplózott tevékenységeit tartalmazza.
   * **Tartalom:** Felhasználói azonosító (`user_id`), anime azonosító (`anime_id`), adott pontszám (`score`), megtekintési státusz (`watching_status`) és a megnézett epizódok száma (`watched_episodes`).

---

### Az elemzés felépítése (Notebook Struktúra)
A Jupyter Notebook a későbbi prezentációs logikát követve 5 fő szakaszra bomlik:

1. **Adatelőkészítés és Technikai Optimalizálás:** Hiányzó és szöveges (`"Unknown"`) értékek kezelése, adattípusok konvertálása, megjelenési évek kinyerése, valamint a 109 millió soros `animelist.csv` memóriahatékony (downcastolt) beolvasása.
2. **A Nagy Háromszög (Minőség vs. Rajongás vs. Népszerűség):** A `Score`, `Favorites` és `Members` mutatók eloszlásának, rangkorrelációjának vizsgálata, új relatív mutatók (pl. *Kedvenc-arány*) bevezetése és az outlierek (kvadránsok) azonosítása.
3. **Háttértényezők szerepe (Típus, Kor, Epizódszám):** A folytatás-torzítás (*sequel bias*), az újdonság-torzítás (*recency bias*) és az elköteleződési hatás (*commitment effect*) statisztikai igazolása.
4. **Mélyfúrás és Egyedi Mutatók:**
   * *Kaszálási és Halogatási Index:* Mely animéket hagyják félbe a legtöbben (`Drop Rate`), és melyek ragadnak be a „megnézendő” (`Plan to Watch`) listákba?
   * *Polarizációs Index:* A legmegosztóbb animék felkutatása a szavazatok szórása (`Score-1`...`Score-10`) alapján.
<!-- 5. **Big Data Insightok (`animelist.csv`):** A híres *„3 epizódos szabály”* empirikus tesztelése a lemorzsolódási adatokon, valamint a *„veterán kritikus-effektus”* vizsgálata (szigorúbban pontoznak-e azok, akik több száz animét láttak?). -->

---
*Az alábbi blokkban a szükséges Python könyvtárak importálásával és a globális megjelenítési beállításokkal indítjuk a munkát.*

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import seaborn as sns


df_anime = pd.read_csv('data/anime.csv')
#df_animelist = pd.read_csv('data/animelist.csv')


### Segédfüggvények


In [ ]:
# Generated by Github Copilot
def duration_to_minutes(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    value = re.sub(r"\s+", " ", str(value)).strip()

    match = re.fullmatch(
        r"(?:(\d+(?:\.\d+)?)\s*hr\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*min\.\s*)?"
        r"(?:(\d+(?:\.\d+)?)\s*sec\.\s*)?"
        r"(?:per ep\.)?",
        value,
    )

    if match is None:
        raise ValueError(f"Unexpected duration format: {value!r}")

    hours, minutes, seconds = match.groups()

    return (
        float(hours or 0) * 60
        + float(minutes or 0)
        + float(seconds or 0) / 60
    )

# Generated by Github Copilot
def premiered_to_year(value):
    if pd.isna(value) or value == "Unknown":
        return pd.NA

    match = re.search(r"\d{4}", str(value).strip())


    if match is None:
        raise ValueError(f"Unexpected premiered format: {value!r}")

    return int(match.group(0))

### Adattisztítás és típuskonverzió (`anime.csv`)

A nyers adathalmazban a hiányzó értékeket szöveges `"Unknown"` karakterlánc jelöli, ami miatt a Pandas számos számszerű oszlopot szövegként (`object` típusként) olvas be. Az alábbi kódrészlet elvégzi az adatok szabványosítását és a későbbi elemzéshez szükséges új változók (feature-ök) kinyerését:

* **Hiányzó értékek és numerikus mezők kezelése:** Az `"Unknown"` értékeket szabványos `pd.NA`-ra cseréljük, majd az értékeléseket (`Score`, `Score-1`...`Score-10`), a rangsort (`Ranked`) és az epizódszámot (`Episodes`) numerikus típusúvá alakítjuk.
* **Időtartam és premier éve:** Segédfüggvények segítségével a szöveges `Duration` oszlopból kinyerjük az epizódok percekben mért hosszát (`Duration_minutes`), a szezonális `Premiered` mezőből pedig a megjelenés évét (`Premiered_year`).
* **Vetítési időszak feldolgozása:** Az `Aired` oszlop intervallumait (`" to "` mentén) szétválasztjuk kezdő (`start_date`) és befejező (`end_date`) dátumokra. Az egynapos megjelenéseknél (pl. mozifilmek, OVA-k) a kezdő és záró dátum megegyezik. A két időpont különbségéből kiszámítjuk a vetítés teljes időtartamát napokban (`length_days`).
* **Ellenőrzés:** A blokk végén egy összesítő táblázatban ellenőrizzük az oszloponkénti hiányzó értékek (`NaN`) számát és a véglegesített adattípusokat.

In [ ]:
#Data uses "Unknown" instead of NaN, so we need to replace it with NaN
df_anime.replace("Unknown", pd.NA, inplace=True)
string_numerics = ["Score", "Episodes", "Ranked", "Score-1", "Score-2", "Score-3", "Score-4", "Score-5", "Score-6", "Score-7", "Score-8", "Score-9", "Score-10"]
for col in string_numerics:
    df_anime[col] = pd.to_numeric(df_anime[col], errors='raise')

df_anime["Duration_minutes"] = (
    df_anime["Duration"]
    .apply(duration_to_minutes)
    .astype("Float64")
)

df_anime["Premiered_year"] = (
    df_anime["Premiered"]
    .apply(premiered_to_year)
    .astype("Int64")
)

#df_anime[["Start_date", "End_date", "Length_days"]] = df_anime["Aired"].apply(parse_air_date)

aired = (
    df_anime["Aired"]
    .astype("string")
    .str.strip()
    .replace(["Unknown", "?"], pd.NA)
)

dates = aired.str.split(" to ", n=1, expand=True)

# A single date is both the start and end date
start_raw = dates[0]
end_raw = dates[1].fillna(dates[0])

# If either side is "?", it becomes NaT
start_date = pd.to_datetime(start_raw, errors="coerce", format="mixed")
end_date = pd.to_datetime(end_raw, errors="coerce", format="mixed")

df_anime["Start_date"] = start_date
df_anime["End_date"] = end_date

# Automatically becomes NaN when either date is missing
df_anime["Length_days"] = (end_date - start_date).dt.days

# Relatív rajongási mutató: a nézők hány százaléka jelölte kedvencnek (%)
df_anime["Favorite_Percentage"] = (df_anime["Favorites"] / df_anime["Members"]) * 100

#df_anime["Actual_Members"] = df_anime["Watching"] + df_anime["Completed"] + df_anime["On-Hold"] + df_anime["Dropped"]

display(pd.DataFrame({
    "NaN count": df_anime.isna().sum(),
    "type": df_anime.dtypes
}))

In [ ]:
numerics = string_numerics + ["Popularity", "Members", "Favorites", "Watching", "Completed", "On-Hold", "Dropped", "Length_days", "Duration_minutes", "Premiered_year", "Favorite_Percentage"]

s_nancounts = df_anime[numerics].isna().sum()

desc = df_anime[numerics].describe().T

desc["NaNCount"] = s_nancounts

desc

# Korreláció

In [ ]:
#ai-usage: prompt1

# Csak az értékeléssel rendelkező és legalább 1000 fő által listázott animéket tartjuk meg
df_corr = df_anime.dropna(subset=['Score', 'Members', 'Favorites', 'Favorite_Percentage']).copy()
df_corr = df_corr[df_corr['Members'] >= 1000].copy()


# Elemzendő oszlopok és magyar címkéik a grafikonokhoz
cols = ['Score', 'Members', 'Favorites', 'Favorite_Percentage']
labels_heatmap = [
    'Átlagpontszám\n(Score)', 
    'Nézői elérés\n(Members)', 
    'Kedvencek száma\n(Favorites)', 
    'Rajongási arány\n(Favorites %)'
]

# Spearman-rangkorrelációs mátrix kiszámítása
spearman_matrix = df_corr[cols].corr(method='spearman')


fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

# Felső háromszög maszkolása az áttekinthetőségért (opcionális, itt teljes mátrixot mutatunk)
sns.heatmap(
    spearman_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=1.5,
    cbar_kws={"label": "Spearman-féle rangkorreláció (ρ)"},
    xticklabels=labels_heatmap,
    yticklabels=labels_heatmap,
    annot_kws={"size": 12},
    ax=ax
)

ax.set_title(
    "A Nagy Háromszög: Minőség, Elérés és Rajongás Korrelációja\n(Spearman ρ, szűrve: Members ≥ 1000)", 
    fontsize=13, fontweight="bold", pad=15
)
plt.xticks(rotation=0, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()


# A vizualizációhoz log10 transzformációt alkalmazunk a ferde eloszlású változókra,
# hogy a scatter plotokon láthatóvá váljon a valódi szerkezet.
df_plot = pd.DataFrame()
df_plot['Score (1-10)'] = df_corr['Score']
df_plot['log10(Members)'] = np.log10(df_corr['Members'])
# Hozzáadunk 1-et a Favorites oszlophoz, hogy elkerüljük a log10(0) problémát
df_plot['log10(Favorites) + 1'] = np.log10(df_corr['Favorites'] + 1)
df_plot['log10(Favorite_Percentage) + 0.01'] = np.log10(df_corr['Favorite_Percentage'] + 0.01)


# PairGrid inicializálása
g = sns.PairGrid(df_plot, diag_sharey=False, corner=True)

# Alsó háromszög: Scatter plotok alacsony átlátszósággal (alpha) a sűrűség érzékeltetésére
g.map_lower(
    sns.scatterplot, 
    alpha=0.15, 
    s=15, 
    color="#1f77b4", 
    edgecolor="none"
)

# Átló: Hisztogramok és sűrűségfüggvények (KDE) az egyes változók eloszlásáról
g.map_diag(
    sns.histplot, 
    kde=True, 
    color="#2b5c8f", 
    bins=60, 
    edgecolor="white",
)

# Főcím beállítása
g.fig.subplots_adjust(top=0.92)
g.fig.suptitle(
    "Tulajdonságok Páronkénti Kapcsolata és Eloszlása (Scatter Plot Mátrix, Members ≥ 1000)", 
    fontsize=15, fontweight="bold"
)

plt.show()

### 1.1. A korrelációs hőtérkép és a páronkénti eloszlások (Pairplot) elemzése

A statisztikailag releváns mezőnyre (`Members >= 1000`) elvégzett Spearman-féle rangkorrelációs elemzés és a logaritmizált páronkénti pontfelhő-mátrix az alábbi főbb összefüggéseket tárja fel a minőség (`Score`), az elérés (`Members`) és a rajongás (`Favorites`, `Favorites %`) között:

#### 1. Korrelációs mintázatok (Spearman)
* **Elérés és nyers rajongás erős összefonódása:** A legerősebb kapcsolat a `Members` és a `Favorites` között mérhető. Ez egy természetes skálázódás hatás: minél több felhasználó vesz fel egy animét a listájára, annál nagyobb a potenciális bázis, amelyből a kedvenc-jelölések kikerülnek. A nyers `Favorites` szám tehát önmagában erősen népszerűség-vezérelt mutató.
* **Minőség vs. Nyers mutatók:** Az átlagpontszám (`Score`) közepesen erős, pozitív korrelációt mutat mind a nézőszámmal (`0.65`), mind a kedvencek számával (`0.68`). A jó minőségű animék tehát általánosságban szélesebb közönséget érnek el.
* **A relatív rajongási arány önállósodása:** Amikor a kedvencek számát megtisztítjuk a nézőszám hatásától (`Favorites % = Favorites / Members * 100`), a `Members`-szel vett korreláció `0.40`-re, a `Score`-ral vett korreláció pedig `0.48`-ra mérséklődik. Ez igazolja, hogy a **kult-faktor (relatív rajongási intenzitás)** egy önálló dimenzió: nem feltétlenül a legnézettebb animék váltják ki a legmélyebb szubjektív elköteleződést.

#### 2. Egyváltozós eloszlások (A Pairplot főátlója)
* **`Score (1-10)`:** Megközelítőleg haranggörbe (normális) eloszlást követ, amelynek csúcsa a `6.5 - 7.2` tartományban található, enyhe bal oldali elnyúlással (a nagyon gyenge, 2–4 pontos címek felé).
* **`log10(Members)` és `log10(Favorites) + 1`:** Az eredeti változók extrém módon jobbra ferdültek, és még a 10-es alapú logaritmikus skálán is erősen aszimmetrikusak. A `log10(Members)` eloszlása `3.0`-nál (az 1000 fős szűrési küszöbnél) magasan kezdődik, majd a milliós nézettségű (`> 6.0`) kasszasikerek felé fokozatosan elvékonyodik.
* **`log10(Favorite_Percentage) + 0.01`:** A fő tömeg egy szabályos haranggörbét alkot `-0.7` körül, míg a `-2.0` értéknél (`log10(0 + 0.01)`) egy éles tüske látható, amely a több mint 1000 nézővel, de **0 darab** kedvenc-jelöléssel rendelkező animéket tömöríti.

#### 3. Nemlineáris kapcsolatok és adatszerkezeti sajátosságok (Alsó háromszög)
* **`Score` vs. `Members`, `Favorites`, `Favorites %`:** Az első oszlop scatter plotjain jól látható, hogy `Score = 7.0` alatt a pontfelhő lapos és alacsony értékek körül sűrűsödik, azonban `7.5` felett a nézettség és a rajongási arány meredeken, nemlineárisan kilő. Ugyanakkor a bal oldalon (`Score < 6.0`, de magas Y érték) megjelennek azok a különleges outlierek, amelyek alacsony átlagpontszámuk ellenére jelentős elérést vagy rajongótábort építettek ki.
* **`log10(Members)` vs. `log10(Favorites) + 1`:** A két változó kapcsolata a magasabb tartományokban (`log10(Members) > 4.5`) rendkívül szoros, lineáris csatornába rendeződik. A tölcsér alsó részén látható vízszintes vonalak a kis egész számokból (`0, 1, 2, 3` kedvenc-jelölés) adódó diszkrét ugrásokat szemléltetik.
* **`log10(Members)` vs. `log10(Favorite_Percentage)`:** A második oszlop legalsó ábrájának bal alsó sarkában párhuzamos, lefelé ívelő diagonális sávok láthatók. Ezek matematikai műtermékek: mivel a `Favorites %` képlete $\frac{\text{Favorites}}{\text{Members}} \times 100$, a `-2.0` szinten húzódó vízszintes vonal pedig a 0 kedvenccel rendelkező címeket jelöli.

# Toplista átfedések

In [ ]:
#Ai-usage: prompt2

df_top = df_corr.copy()

# Top 50 halmazok definiálása
top_n = 50

top_score = set(df_top.nlargest(top_n, "Score")["MAL_ID"])
top_members = set(df_top.nlargest(top_n, "Members")["MAL_ID"])
top_favs = set(df_top.nlargest(top_n, "Favorites")["MAL_ID"])
top_fav_ratio = set(df_top.nlargest(top_n, "Favorite_Percentage")["MAL_ID"])

top_sets = {
    "Top 50 Értékelés (Score)": top_score,
    "Top 50 Népszerűség (Members)": top_members,
    "Top 50 Rajongás (Favorites)": top_favs,
    "Top 50 Rajongási Arány (Fav/Mem)": top_fav_ratio,
}

# 1. Átfedési mátrix (Közös elemek száma és % aránya)
matrix_names = list(top_sets.keys())
overlap_counts = np.zeros((len(matrix_names), len(matrix_names)), dtype=int)

for i, name1 in enumerate(matrix_names):
    for j, name2 in enumerate(matrix_names):
        overlap_counts[i, j] = len(top_sets[name1].intersection(top_sets[name2]))

overlap_df = pd.DataFrame(
    overlap_counts, index=matrix_names, columns=matrix_names
)

# Vizualizáció: Heatmap a PPT-hez
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    overlap_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar_kws={"label": "Közös animék száma (max 50)"},
    linewidths=1,
    annot_kws={"size": 13, "weight": "bold"},
    ax=ax,
)

ax.set_title(
    f"Top {top_n} Toplisták Páronkénti Átfedése (Közös címek száma)", pad=15
)
plt.xticks(rotation=20, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 2. A "Szent Grál" animék: Amelyek mindhárom alaplistán (Score, Members, Favorites) Top 50-esek
all_three_core = top_score & top_members & top_favs
df_elit = df_top[df_top["MAL_ID"].isin(all_three_core)][
    [
        "Name",
        "Type",
        "Premiered_year",
        "Start_date",
        "Score",
        "Members",
        "Favorites",
        "Favorite_Percentage",
    ]
].sort_values(by="Score", ascending=False)

print(
    f"\nMindhárom alapmutatóban (Score, Members, Favorites) Top {top_n} anime ({len(df_elit)} db):"
)
display(df_elit.reset_index(drop=True))

### Elemzési Észrevételek: Top 50 Listák Átfedése és a „Elit” Csoport

Az aggregált mérőszámok összehasonlítása egyértelműen rámutat a minőség (`Score`), az elérés (`Members`) és az érzelmi elköteleződés (`Favorites` / arány) közötti dimenziók strukturális eltéréseire.

---

#### 1. Páronkénti Átfedések Vizsgálata (Heatmap Elemzés)

* **Erős kapcsolódás a volumenalapú metrikák között (Members vs. Favorites – 32 / 50 közös cím):**
  A legmagasabb átfedés a népszerűség és az abszolút kedvencek listája között figyelhető meg. Ez nem meglepő, mivel egy nagyobb felhasználói bázisból könnyebben gyűlik össze magas abszolút kedvenc-szám, még visszafogottabb konverziós arány mellett is.
* **Kritikai elismerés vs. Tömeges elérés (Score vs. Members – mindössze 10 / 50 közös cím):**
  A legkisebb átfedést a pontszám és a nézőszám mutatja. A legjobbra értékelt 50 cím 80%-a nem tartozik a legnézettebb 50 anime közé. Ez a jelenség a rétegműfajok magas értékeléséből, valamint a későbbi évadok túlélési torzításából (sequel bias) fakad, ahol a lemorzsolódott casual nézők már nem húzzák le az átlagot.
* **A rajongási arány függetlenítő szerepe (Fav/Mem arány):**
  A relatív rajongási arány (`Favorites / Members`) mind az értékeléssel (19 cím), mind a kedvencek abszolút számával (25 cím) kiegyensúlyozottabb kapcsolatban áll, mint a nyers eléréssel (16 cím). Ez megerősíti, hogy a normalizált arány képes kiszűrni a pusztán a nagy nézőszámból fakadó torzításokat.

---

#### 2. A Konszenzusos Elit („Elit” – Mindhárom Alapmutatóban Top 50)

A 17 562 animét számláló teljes adatbázisból mindössze **9 cím** képes egyszerre a legmagasabbra értékelt, a legnézettebb és a legtöbbször kedvencnek jelölt 50 anime közé kerülni:

* **Tökéletes konszenzus és elköteleződés:**
  A lista élén álló trió (*Fullmetal Alchemist: Brotherhood*, *Steins;Gate*, *Hunter x Hunter (2011)*) nem csupán a kritikai ranglétrát vezeti 9.10 feletti pontszámmal, de a nézőbázisuk több mint 8%-a kedvencnek is jelölte őket, ami a leghíresebb animék között is kiemelkedően magas konverziós arány.
* **Formátumbeli dinamika (Filmek jelenléte):**
  A listára két egész estés mozifilm (*Koe no Katachi*, *Kimi no Na wa.*) is felkerült. Bár ezek pontszáma eléri vagy megközelíti a 9.00-t, rajongási arányuk (~4.1–4.5%) elmarad a hosszabb sorozatokétól. Ez alátámasztja az elköteleződési hipotézist (commitment effect): a filmek rövidebb játékideje alatt nehezebb olyan mély kötődést kialakítani, mint egy több tíz epizódon átívelő TV-sorozat esetén.
* **Időtállóság és franchise-erő:**
  A *Code Geass* mindkét évadával képviselteti magát a szűk elitben, ami a 2000-es évek közepének ritka, fenntartható sikerességét bizonyítja. A *Cowboy Bebop* jelenléte pedig igazolja a nosztalgia-faktort: 1998-as bemutatója ellenére mindmáig megőrizte pozícióját mindhárom dimenzió élmezőnyében.

In [ ]:
#Ai-usage: prompt2

# Küszöbértékek dinamikus számítása kvantilisekkel a df_clean (Members >= 10,000) halmazon
q_mem_95 = df_top["Members"].quantile(0.95)
q_mem_90 = df_top["Members"].quantile(0.90)
q_mem_50 = df_top["Members"].quantile(0.50)

q_score_90 = df_top["Score"].quantile(0.90)
q_score_40 = df_top["Score"].quantile(0.40)

q_favratio_98 = df_top["Favorite_Percentage"].quantile(0.98)
q_favratio_30 = df_top["Favorite_Percentage"].quantile(0.30)


def classify_outliers(row):
    mem = row["Members"]
    score = row["Score"]
    fav_r = row["Favorite_Percentage"]

    # 1. Overhyped / Guilty Pleasure Blockbuster
    if mem >= q_mem_95 and score <= q_score_40:
        return "1. Overhyped (Magas Members, Alacsony Score)"

    # 2. Réteg-remekmű (Hidden Gem) - kizárjuk a még nem befejezett vagy nagyon kis mintás címeket
    elif score >= q_score_90 and mem <= 80000:
        return "2. Réteg-remekmű (Magas Score, Alacsony Members)"

    # 3. Kultikus Klasszikus (Die-hard Fanbase)
    elif fav_r >= q_favratio_98 and mem < q_mem_95:
        return "3. Kultikus Klasszikus (Extrém Fav/Members arány)"

    # 4. Passzív Közönségsiker (Mindenki látta, kevesek kedvence)
    elif mem >= q_mem_90 and fav_r <= q_favratio_30:
        return "4. Passzív Közönségsiker (Magas Members, Alacsony Fav%)"
    
    else:
        return "Normál"


df_top["Outlier_Category"] = df_top.apply(classify_outliers, axis=1)

# Csoportok elemszámának és átlagos mutatóinak áttekintése
summary_outliers = (
    df_top.groupby("Outlier_Category")
    .agg(
        Darabszám=("MAL_ID", "count"),
        Átlag_Score=("Score", "mean"),
        Medián_Members=("Members", "median"),
        Átlag_Fav_Arány_Pct=("Favorite_Percentage", "mean"),
    )
    .round(2)
    .sort_values(by="Darabszám", ascending=True)
)

display(summary_outliers)

# Top 5 példa kiírása kategóriánként a PPT narratívához
for cat in sorted(df_top["Outlier_Category"].unique()):
    if cat == "Normál / Átlagos eloszlás":
        continue
    #print(f"\n--- {cat} (Top példák) ---")
    subset = df_top[df_top["Outlier_Category"] == cat]

    # Kategóriától függő rendezés a legjellemzőbb példákhoz
    if "1." in cat or "4." in cat:
        subset = subset.sort_values(by="Members", ascending=False)
    elif "2." in cat:
        subset = subset.sort_values(by="Score", ascending=False)
    else:
        subset = subset.sort_values(by="Favorite_Percentage", ascending=False)

    #display(subset[["Name","Type","Episodes","Premiered_year","Score","Members","Favorite_Percentage",]].head(6))

In [ ]:
#Ai-usage: prompt2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

ax1: plt.Axes = ax1
ax2: plt.Axes = ax2

# Színpaletta az outlier csoportoknak (a normál pontok halványszürkék a háttérben)
palette = {
    "Normál": "#d3d3d3",
    "1. Overhyped (Magas Members, Alacsony Score)": "#e74c3c",  # Piros
    "2. Réteg-remekmű (Magas Score, Alacsony Members)": "#2ecc71",  # Zöld
    "3. Kultikus Klasszikus (Extrém Fav/Members arány)": "#9b59b6",  # Lila
    "4. Passzív Közönségsiker (Magas Members, Alacsony Fav%)": "#f39c12",  # Narancs
}

left_categories = [
    "1. Overhyped (Magas Members, Alacsony Score)",
    "2. Réteg-remekmű (Magas Score, Alacsony Members)",
]
right_categories = [
    "3. Kultikus Klasszikus (Extrém Fav/Members arány)",
    "4. Passzív Közönségsiker (Magas Members, Alacsony Fav%)",
]

# Bal DF: minden, ami nem 1-es vagy 2-es kategória, "Normál" lesz
df_ax1 = df_top.copy()
df_ax1["Plot_Category"] = df_ax1["Outlier_Category"].where(
    df_ax1["Outlier_Category"].isin(left_categories), "Normál"
)
df_ax1 = df_ax1.sort_values(
    by="Plot_Category", key=lambda x: x != "Normál"
)

# Jobb DF: minden, ami nem 3-as vagy 4-es kategória, "Normál" lesz
df_ax2 = df_top.copy()
df_ax2["Plot_Category"] = df_ax2["Outlier_Category"].where(
    df_ax2["Outlier_Category"].isin(right_categories), "Normál"
)
df_ax2 = df_ax2.sort_values(
    by="Plot_Category", key=lambda x: x != "Normál"
)


# Rendezés, hogy a színes outlier pontok a szürke pontok FÖLÉ kerüljenek a rajzolásnál
df_plot = df_top.sort_values(
    by="Outlier_Category", key=lambda x: x != "Normál"
)

# Members (Log) vs. Score
sns.scatterplot(
    data=df_ax1,
    x="Members",
    y="Score",
    hue="Plot_Category",
    hue_order=["Normál"] + left_categories,
    palette=palette,
    alpha=0.75,
    s=45,
    ax=ax1,
    legend=False,
)
ax1.set_xscale("log")
ax1.set_title("Népszerűség vs. Kritikai Értékelés (Outlier Kvadránsok)")
ax1.set_xlabel("Nézői érdeklődés - Members (Logaritmikus skála)")
ax1.set_ylabel("Átlagos értékelés (Score)")
#ax1.axhline(q_score_90, color="grey", linestyle="--", alpha=0.5)
#ax1.axvline(q_mem_95, color="grey", linestyle="--", alpha=0.5)
ax1.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{int(x):,}"))

# Favorite_Percentage vs. Members (Log)
sns.scatterplot(
    data=df_ax2,
    x="Members",
    y="Favorite_Percentage",
    hue="Plot_Category",
    hue_order=["Normál"] + right_categories,
    palette=palette,
    alpha=0.75,
    s=45,
    ax=ax2,
)
ax2.set_title("Népszerűség vs. Relatív Rajongás (Favorites / Members %)")
ax2.set_ylabel("Rajongási arány - Favorites / Members (%)")
ax2.set_xlabel("Nézői érdeklődés - Members (Logaritmikus skála)")
ax2.set_xscale("log")
#ax2.axhline(q_favratio_98, color="grey", linestyle="--", alpha=0.5)

# Jelmagyarázat elhelyezése az ábra alatt, prezentáció-barát módon
handles, labels = ax2.get_legend_handles_labels()
legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label=label,
        markerfacecolor=color,
        markersize=8,
        alpha=0.85,
    )
    for label, color in palette.items()
]

ax2.get_legend().remove()
fig.legend(
    handles=legend_handles,
    #labels,
    loc="lower center",
    ncol=3,
    frameon=True,
    fontsize=10,
    bbox_to_anchor=(0.5, -0.08),

)

plt.suptitle(
    "A Nagy Háromszög Szétválása: Karakteres Anime Archetípusok (Members >= 10 000)",
    fontsize=16,
    weight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()